In [1]:
from tensorflow.keras.datasets import mnist
import numpy as np
from Seera_init import tensor as Tensor
import matplotlib.pyplot as plt
from Seera import (
    Input, Conv2D, MaxPool2D, Flatten, Dense,
    Sequential, Loss, Adam,
)
from Seera_Engine import autograd4nn


2026-04-11 02:01:17.400907: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-11 02:01:17.438802: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-11 02:01:18.149242: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train_ = np.expand_dims(X_train,axis=1)/255
y_train_ = np.zeros((60000,10))
for i in range (0,60000):
    y_train_[i,:] = np.eye(1,10,y_train[i])

In [3]:
model = Sequential([
    Input((1, 28, 28)),
    Conv2D(8, 1, (3, 3), activation="relu", stride=1, zero_padding=1),
    MaxPool2D(pool_size=(2, 2), stride=2),
    Conv2D(16, 8, (3, 3), activation="relu", stride=1, zero_padding=1),
    MaxPool2D(pool_size=(2, 2), stride=2),
    Flatten(),
    Dense(7*7*4*4, 128, activation="relu"),
    Dense(128, 16, activation="relu"),
    
    Dense(16, 10, activation="softmax"),
], device="cuda")
model.summary()

Model Summary
  Layer 0: Input Layer with shape (1, 28, 28)
  Layer 1: Conv2D(1→8, kernel=(3, 3), act=relu)
  Layer 2: MaxPool2D(pool=(2, 2), stride=(2, 2))
  Layer 3: Conv2D(8→16, kernel=(3, 3), act=relu)
  Layer 4: MaxPool2D(pool=(2, 2), stride=(2, 2))
  Layer 5: Flatten Layer
  Layer 6: Dense(784→128, act=relu, params=100480)
  Layer 7: Dense(128→16, act=relu, params=2064)
  Layer 8: Dense(16→10, act=softmax, params=170)


In [4]:
loss_fn = Loss()
optimizer = Adam(model, lr=1e-4)

In [5]:
idx = np.random.permutation(len(X_train_))
X, y = X_train_[idx], y_train_[idx]  # shuffle (assuming labels y)

In [ ]:

batch_size = 16
epochs = 3  
loss_track = 0.0
for epoch in range(epochs):
    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, 60000, batch_size):
        x_batch, y_batch = X[i:i+batch_size], y[i:i+batch_size]
        X_batch = Tensor(x_batch, is_leaf=True, device="cuda")
        Y_batch = Tensor(y_batch, device="cuda")
        
        pred = model.forward(X_batch)
        loss = loss_fn.categorical_cross_entropy(pred, Y_batch)
        print(X_batch.value.to_host_f32().sum())
        print("true",end="")
        print(x_batch.sum())
        
        if(float(loss.value.to_host_f32().ravel()[0])==0):
            print("#########down######")
            # print(X_batch)  
            
            print(X_batch.value.to_host_f32().sum())
            print(Y_batch.value.to_host_f32())
        loss_val = float(loss.value.to_host_f32().ravel()[0])
        epoch_loss += loss_val
        n_batches += 1
        # print(Y_batch.sum())
        
        del(X_batch)
        del(Y_batch)

        model.zero_grad()
        autograd4nn(loss)
        optimizer.step()
        
        
    avg_loss = epoch_loss / n_batches
    print(f"EPOCH {epoch+1}/{epochs}: Loss: {avg_loss:.10f}")

1706.851
true1706.8509803921568
1401.3334
true1401.3333333333335
1283.698
true1283.6980392156863
1491.353
true1491.3529411764707
1600.5254
true1600.5254901960784
1570.4706
true1570.4705882352941
1607.5922
true1607.592156862745
1416.0627
true1416.0627450980392
1538.0039
true1538.0039215686274
1693.1099
true1693.1098039215685
1675.0509
true1675.0509803921568
1708.7295
true1708.729411764706
1587.8981
true1587.8980392156861
1718.5255
true1718.5254901960784
1769.3726
true1769.3725490196077
1551.9686
true1551.9686274509804
1517.8628
true1517.8627450980393
1695.8472
true1695.8470588235296
1697.0039
true1697.0039215686274
1470.0785
true1470.078431372549
1636.2903
true1636.2901960784313
1674.2628
true1674.2627450980392
1653.2118
true1653.2117647058824
1552.0471
true1552.0470588235294
1574.3412
true1574.3411764705882
1547.145
true1547.1450980392158
1546.8589
true1546.858823529412
1509.5726
true1509.5725490196078
1730.1765
true1730.1764705882354
2036.0079
true2036.007843137255
1512.604
true1512.6

In [21]:
X_test_ = np.expand_dims(X_test,axis=1)/255

correct = 0
for i in range(len(X_test)):
    x = Tensor(X_test_[i:i+1], is_leaf=True, device="cuda")
    pred = model.predict(x)
    # bring to host for argmax
    # print(type(pred.value))
    pred_np = pred.value.to_host_f32()
    pred_label = np.argmax(pred_np)
    if pred_label == y_test[i]:
        correct += 1

accuracy = correct / len(X_test) * 100
print(f"\nTest Accuracy: {accuracy:.1f}% ({correct}/{len(X_test)})")
print("GPU test complete ✓")


Test Accuracy: 97.9% (9787/10000)
GPU test complete ✓


In [ ]:
batch_size = 16
epochs = 5  
loss_track = 0.0
for epoch in range(epochs):
    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, 60000, batch_size):
        X_batch, y_batch = X[i:i+batch_size], y[i:i+batch_size]
        X_batch = Tensor(X_batch, is_leaf=True, device="cuda")
        Y_batch = Tensor(y_batch, device="cuda")
        
        print(Y_batch.sum())
    print("FONE#####################################################################################")
        
    

In [ ]:


from tensorflow.keras.datasets import mnist
import numpy as np
from Seera_init import tensor as Tensor
import matplotlib.pyplot as plt
from Seera import (
    Input, Conv2D, MaxPool2D, Flatten, Dense,
    Sequential, Loss, Adam,
)
from Seera_Engine import autograd4nn

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train_ = np.expand_dims(X_train,axis=1)/255
y_train_ = np.zeros((60000,10))
for i in range (0,60000):
    y_train_[i,:] = np.eye(1,10,y_train[i])
# y_train_ = y_train_[:,:,np.newaxis]
X_test_ = np.expand_dims(X_test,axis=1)/255
print("=" * 60)
print("  MNIST GPU Test — Seera Framework (cuda)")
print("=" * 60)

print(f"Train: {X_train.shape}, Test: {y_test.shape}")

# ─── Build Model (device="cuda") ─────────────────────────
model = Sequential([
    Input((1, 28, 28)),
    Conv2D(8, 1, (3, 3), activation="relu", stride=1, zero_padding=1),
    MaxPool2D(pool_size=(2, 2), stride=2),
    Conv2D(16, 8, (3, 3), activation="relu", stride=1, zero_padding=1),
    MaxPool2D(pool_size=(2, 2), stride=2),
    Flatten(),
    Dense(16*7*7, 128, activation="relu"),
    Dense(128, 16, activation="relu"),
    
    Dense(16, 10, activation="softmax"),
], device="cuda")
model.summary()
# model = Sequential([
#     Input((1,28,28,)),
#     Flatten(),
#     Dense(784, 256, activation="relu"),
#     Dense(256, 128, activation="relu"),
#     Dense(128, 10, activation="softmax"),
# ], "cuda")

loss_fn = Loss()
optimizer = Adam(model, lr=1e-4)
# ─── Train ───────────────────────────────────────────────
# loss_fn = Loss()
# optimizer = Adam(model, lr=0.001)


idx = np.random.permutation(len(X_train_))
X, y = X_train_[idx], y_train_[idx]  # shuffle (assuming labels y)

batch_size = 16
epochs = 3  
loss_track = 0.0
for epoch in range(epochs):
    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, 25000, batch_size):
        X_batch, y_batch = X[i:i+batch_size], y[i:i+batch_size]
        X_batch = Tensor(X_batch, is_leaf=True, device="cuda")
        Y_batch = Tensor(y_batch, device="cuda")
        pred = model.forward(X_batch)
        loss = loss_fn.categorical_cross_entropy(pred, Y_batch)
        if(i%(1600) ==0 ):
            print(y_batch)
            print("#########down######")
            print(Y_batch.value.to_host_f32())
        # del(X_batch)
        # del(Y_batch)
        loss_val = float(loss.value.to_host_f32().ravel()[0])
        epoch_loss += loss_val
        n_batches += 1

        model.zero_grad()
        autograd4nn(loss)
        optimizer.step()
    # exit()
        
    avg_loss = epoch_loss / n_batches
    print(f"EPOCH {epoch+1}/{epochs}: Loss: {avg_loss:.10f}")
# history = model.fit(
#     X_train_, y_train_,
#     Optimizer=optimizer,
#     Loss=loss_fn.categorical_cross_entropy,
#     Epochs=5,
#     batch_size=16,
#     Loss_interval=1,
# )

# ─── Evaluate ────────────────────────────────────────────
correct = 0
for i in range(len(X_test)):
    x = Tensor(X_test_[i:i+1], is_leaf=True, device="cuda")
    pred = model.predict(x)
    # bring to host for argmax
    pred_np = pred.value.to_host_f32()
    pred_label = np.argmax(pred_np)
    if pred_label == y_test[i]:
        correct += 1

accuracy = correct / len(X_test) * 100
print(f"\nTest Accuracy: {accuracy:.1f}% ({correct}/{len(X_test)})")
print("GPU test complete ✓")